# 02 – Exploratory data analysis of the medical datasets
**Project 2 – Ensemble learning for disease prediction** (reference article: Mahajan et al., 2024)

## Purpose
Before comparing ensemble methods, each dataset is audited for the properties that condition a fair comparison:

1. size, feature types and **class balance**;
2. **missing values**;
3. **duplicate rows**, and the risk that identical records fall on both sides of a train/test split;
4. features that **predict the target almost alone** (possible leakage or a trivially separable problem);
5. the score reached by **default models**, compared with the majority-class rule, to estimate how much room exists to separate methods.

In every dataset the positive class (`y = 1`) is the **disease** class, so that precision, recall and F1 refer to the diseased patients.

## Scope of this run
The project registers the 16 datasets of Table 2 of the reference article. In this run, seven datasets from the UCI repository are analysed (D2, D5, D6, D8, D11, D12, D16).
The Kaggle datasets that are binary (D1, D3, D4, D7, D9, D10, D13, D14) were not present in `data/raw/` when the notebook was executed, and D15 (HAM10000, multiclass) is handled separately;
none of these is covered by the interpretations below. D2 and D12 are the UCI versions of the Kaggle files used in the article (same patients).
All random operations use the seed defined in `configs/config.yaml`.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from ensemble_disease.config import load_config, set_seed
from ensemble_disease.data import REGISTRY
from ensemble_disease import eda

cfg = load_config(); SEED = cfg["seed"]; set_seed(SEED)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
eda.RESULTS.mkdir(exist_ok=True)
print("seed =", SEED)

## 1. Loading the datasets
Each dataset is loaded as a pair `(X, y)`. Text cells are stripped, `?` and empty strings become missing values, and numeric columns stored as text are converted back to numbers.

In [ ]:
data, errors = eda.load_all()
print(f"Loaded {len(data)}/{len(REGISTRY)} datasets:", list(data))
if errors:
    print("\nDatasets not loaded (source file not found or unreadable):")
    for k, v in errors.items():
        print(f"  - {k}: {v[:110]}")

## 2. Definition of the target (`y = 1`)

In [ ]:
pd.DataFrame({k: dict(positive_class_rule=v["note"]) for k, v in REGISTRY.items()}).T

**Interpretation.** For five datasets the positive class follows directly from the documentation (Cleveland: `num > 0`; ILPD: liver patient; early-stage diabetes: `Positive`;
Debrecen: signs of retinopathy; CKD: `ckd`). Two definitions are conventions rather than clinical labels and must be stated as such in the report:

* **D5 (BUPA liver disorders)** – the label is the `selector` field, which the UCI documentation describes as a field used to split the data into two sets, not a clinical diagnosis. The `drinks` variable is kept as an ordinary feature.
* **D16 (dermatology)** – the original problem has six classes. It is binarised here as *psoriasis (class 1) versus the five other erythemato-squamous diseases*; results are therefore not comparable with multi-class accuracies.

## 3. Summary of the datasets

In [ ]:
summary = eda.summary_table(data)
summary.to_csv(eda.RESULTS / "eda_summary.csv")
display(summary)

Definitions: `imbalance_ratio` = majority / minority class size; `duplicate_rows` = rows whose features are an exact copy of an earlier row; `rows_with_missing` = rows with at least one missing cell.

In [ ]:
eda.plot_class_balance(summary); plt.show()

**Interpretation.**

* **Size and dimensionality.** The datasets are small: between 303 (Cleveland) and 1,151 (Debrecen) patients, with 6 to 34 features. At this scale the variance of a cross-validated score is not negligible, so a single train/test split would be unreliable and repeated cross-validation is required.
* **Feature types.** D8 is almost entirely categorical (15 Yes/No symptoms plus age) and D12 mixes 14 numeric and 10 categorical variables; the other datasets are numeric (D6 has one categorical variable, `Gender`). Preprocessing must therefore include categorical encoding, fitted inside each training fold.
* **Class balance.** No dataset is severely imbalanced: the largest ratios are 2.49 (D6, 71.4 % patients) and 2.27 (D16 binarised, 30.6 % positives); the others lie between 1.13 and 1.67. Accuracy remains interpretable, but it is reported together with recall and F1, because a missed patient (false negative) is clinically more costly than a false alarm.
* **Where the majority rule stands.** The proportion of the largest class (from 54 % in D2 to 71 % in D6) is the accuracy of a classifier that ignores the features; any model must be compared with this value (section 9).

## 4. Missing values

In [ ]:
display(summary[["n", "missing_cells", "missing_pct", "rows_with_missing"]])
eda.plot_missing(data); plt.show()

**Interpretation.** Missing data is a marginal problem everywhere except in **D12 (chronic kidney disease)**.

* D2, D6 and D16 have between 4 and 8 missing cells (below 0.2 % of the cells), each in a distinct row; D5, D8 and D11 have none.
* D12 has 1,012 missing cells (10.5 %) spread over **242 of its 400 patients (60.5 %)**. Dropping incomplete rows would discard most of the dataset, so imputation is unavoidable. The figure shows that the gaps concentrate in a few laboratory variables: red blood cells (`rbc`, about 38 %), red blood cell count (`rbcc`, about 33 %), white blood cell count (`wbcc`, about 27 %), potassium and sodium (about 22 %), and also in `hemo` (about 13 %), which is the most predictive variable of the dataset (section 8).
* Methodological consequence: imputation (median for numeric, most frequent for categorical) is implemented inside the cross-validation pipeline so that it is fitted on the training folds only. Imputing before splitting would let test-fold information leak into the training data.

## 5. Comparison with Table 2 of the reference article
The sample sizes, attribute counts and class counts reported in Table 2 of the article (stored in the dataset registry) are compared with those of the versions used here.
`diff = ours − article`. In the article, the column *Positive/Negative* lists the positive count first.

In [ ]:
ARTICLE = {k: REGISTRY[k]["article"] for k in data}
cmp_ = eda.compare_with_article(summary, ARTICLE)
cmp_.to_csv(eda.RESULTS / "eda_article_comparison.csv", index=False)
display(cmp_.pivot(index="dataset", columns="quantity", values="diff").fillna("").rename(columns=lambda c: f"diff_{c}"))
display(cmp_)

**Interpretation.**

* **Same data versions.** The number of patients is identical to the article for all seven datasets (303, 345, 583, 520, 1,151, 400, 366), so the files used here are the same as the article's.
* **Attribute counts.** They match for D2, D6 and D8 and differ by one for D5 (article 7, here 6), D11 (18 versus 19), D12 (25 versus 24) and D16 (35 versus 34). A plausible explanation is that the article counts the label column (D5, D16) or an identifier column present in the Kaggle file (D12); this is a hypothesis to be stated as such.
* **The definition of the positive class differs for four datasets.** The counts match for D8 and D11, but for **D5, D6 and D12 the article's positive and negative counts are exactly the reverse of ours** (D6: 167/416 in the article, 416/167 here; D12: 150/250 versus 250/150; D5: 200/145 versus 145/200), and D2 is almost reversed (165/138 versus 139/164, a one-patient difference). In the article, the "positive" class is therefore the *non-disease* class for D6 (non-patients) and D12 (`notckd`). This is consistent with a label encoder that gives code 1 to the last label in alphabetical or numerical order, and the very low precision and recall of the article's decision tree and bagging on D6 (35 % and 20 %, Tables 5 and 7) are what is expected when the positive class is the minority healthy group.
* **Consequence for the comparison.** Precision, recall, F1 and AUPRC depend on which class is called positive, so they are **not comparable** with the article on D2, D5, D6 and D12. **Accuracy and ROC-AUC do not depend on this choice** and are the metrics used for any direct comparison with the article.

**Reading the article's Table 3 (accuracy) in light of this analysis.**

* D7 and D12 reach 100 % for all 15 variants, D13 to D15 are at or near 100 %, and D16 gives 96 % for every variant: on these datasets the models cannot be ranked, which agrees with the ceiling effect measured here on D12 and D16 (section 9).
* Two datasets show a jump of more than 20 points for a single variant: **D4** (classical stacking 99 % against 68–77 % for the others) and **D10** (classical stacking 97 % against 85 % for nearly all other variants). Such isolated gains on small datasets are unusual and deserve a check of the evaluation procedure (for instance how the stacking meta-model is trained and tested) before they are taken as evidence of the superiority of stacking.
* The article describes one train/validation/test split with hyper-parameters tuned by 5-fold grid search, and reports no standard deviation or statistical test, so differences of one or two points between variants cannot be interpreted.

## 6. Column types and categorical values
The distinct values of every text column are listed to detect typing errors, case or whitespace variants and unexpected codes.

In [ ]:
for name, (X, y) in data.items():
    print(f"\n===== {name}   shape={X.shape}")
    rep = eda.categorical_report(X)
    if len(rep):
        display(rep)
    else:
        print("   (all features numeric)")

**Interpretation.** After cleaning, every categorical variable has consistent spellings and exactly two levels (`Male/Female`, `Yes/No`, `normal/abnormal`, `present/notpresent`, `good/poor`): there are no typing variants and no unexpected codes, and these variables can be one-hot encoded without further treatment.
Two facts are worth noting for the discussion of results: the sex ratio is unbalanced in D6 (441 men, 142 women, i.e. 76 % men), and D8 is composed of binary symptoms for which many patients share the same answer profile, which anticipates the duplication problem measured below.

## 7. Leakage check A – identical records in the training and test sets

In [ ]:
leak = pd.DataFrame({k: eda.duplicate_leakage(X, y, seed=SEED) for k, (X, y) in data.items()}).T
leak.to_csv(eda.RESULTS / "eda_duplicate_leakage.csv")
display(leak)

`test_rows_with_train_copy_pct` is the percentage of test rows that have an identical copy in the training set, for a random stratified 80/20 split averaged over 50 repetitions.
`dup_groups_conflicting_labels` counts groups of identical records that carry different labels (irreducible label noise).

**Interpretation.**

* **D8 is the critical case.** 269 of its 520 rows (51.7 %) are duplicates of another row, which leaves only **251 distinct patient profiles**. With a random split, **68.9 % of test patients have an exact copy in the training set**. A model that merely memorises its training data therefore scores highly on such a test set. Accuracies measured on D8 with a standard random split are optimistic and cannot be read as generalisation performance.
* The duplicated groups never carry conflicting labels (0 conflicting groups in every dataset), so the copies are consistent repetitions, not label noise; removing them does not discard contradictory information.
* The effect is small elsewhere: 3.6 % of test rows for D6, 1.7 % for D5, 0.7 % for D11, and none for D2, D12 and D16.
* Methodological consequence: the experiments are run under two protocols, one on the data as downloaded and one after removal of exact duplicates, so that the inflation caused by duplicates can be measured.

## 8. Leakage check B – predictive power of a single feature

In [ ]:
auc_tables = {k: eda.single_feature_auc(X, y) for k, (X, y) in data.items()}
top = pd.DataFrame({k: dict(zip(["top1", "top2", "top3"],
        [f"{r.feature} ({r.strength})" for r in t.head(3).itertuples()])) for k, t in auc_tables.items()}).T
display(top)

`strength = max(AUC, 1 − AUC)` is the ROC-AUC obtained when a single feature is used alone as a score (0.5 = no information, 1.0 = perfect separation).
For categorical variables the AUC is computed on the category codes; for binary variables the symmetric definition of `strength` makes the coding order irrelevant.

**Interpretation.** The datasets fall into three groups.

* **One variable almost sufficient.** D16: `clubbing of the rete ridges` (0.984) and `thinning of the suprapapillary epidermis` (0.981) separate psoriasis from the other diseases almost perfectly by themselves; these are histopathological findings that define the disease. D12: `hemo` (0.967), `pcv` (0.952) and `sc` (0.922), the haemoglobin, packed cell volume and serum creatinine, are clinically plausible markers of kidney failure. These are consequences of the disease rather than data-entry leakage, but they make the problem easy.
* **Clinically classical symptoms.** D8: `polyuria` (0.842) and `polydipsia` (0.832) are the two cardinal symptoms of diabetes. The signal is strong but not decisive, and it is spread over several symptoms.
* **Weak individual signals.** D5 (best 0.628), D11 (0.663) and D6 (0.697) have no informative single variable, and D2 (0.764 for `thal`, 0.754 for `cp`, 0.749 for `ca`) has a moderate one. In these datasets the prediction has to rely on combinations of variables, which is the situation where ensembles are expected to help.

## 9. Default models compared with the majority-class rule

In [ ]:
base = pd.DataFrame({k: eda.quick_baselines(X, y, seed=SEED) for k, (X, y) in data.items()}).T
base.to_csv(eda.RESULTS / "eda_baselines.csv")
display(base)

Protocol: 5-fold stratified cross-validation (one repetition) of a majority-class rule, a default decision tree and a default random forest (200 trees). Median imputation and one-hot encoding are inside the pipeline.
This is a diagnostic of the data, not the final experimental protocol. `_acc` = accuracy, `_f1` = F1 of the positive class, `_std` = standard deviation over the folds.

**Interpretation.**

* **Ceiling effects.** On D16 (forest 1.000, tree 0.995), D12 (0.992 and 0.975) and D8 (0.979 and 0.965, with duplicates kept) the models are close to perfect. When every model scores above 0.96, differences between ensemble methods can only be a few thousandths and will not be statistically distinguishable; these datasets cannot rank the methods.
* **Informative datasets.** On D2, D5 and D11 the forest improves clearly on the majority rule (0.835 versus 0.541, 0.722 versus 0.580 and 0.678 versus 0.531) and also on the single tree by 5.8 to 11.0 accuracy points (D2 +7.0, D5 +11.0, D11 +5.8). This is the expected variance-reduction benefit of ensembling, and these datasets are the ones in which a comparison of methods can be informative.
* **D6 (ILPD) is the hardest case.** The forest accuracy (0.710) does not exceed that of the majority rule (0.714), the tree is worse (0.631), and the F1 of the forest (0.809) is below that of the rule that predicts "patient" for everybody (0.833). With default settings the models extract no usable signal beyond the class prior; threshold-free metrics (ROC-AUC, AUPRC) and tuned or class-weighted models will be needed to see whether any signal exists.
* **Accuracy alone can mislead.** The majority rule has an F1 of 0 on D2, D5 and D16, where the majority class is the negative one: its accuracy (0.541, 0.580, 0.694) looks acceptable while it detects no patient. This justifies reporting precision, recall and F1 next to accuracy.
* **Variability.** The standard deviation of the fold accuracies is 0.045 for the forest on D2 and 0.066 on D5, against 0.006 on D12 and 0.000 on D16. On the small, difficult datasets, differences of a few points between two methods lie within the fold-to-fold noise, which motivates repeated cross-validation and statistical tests.

## 10. Correlations between numeric features

In [ ]:
for name, (X, y) in data.items():
    eda.plot_corr(X, y, name); plt.show()

**Interpretation.** Several datasets contain groups of strongly correlated variables, and the correlation with the target is generally modest.

* **D6 (ILPD):** total and direct bilirubin (`TB`, `DB`) are strongly correlated (about 0.87), as are the two transaminases (`Sgpt`, `Sgot`, about 0.8) and total protein with albumin (`TP`, `ALB`, about 0.8). No variable exceeds roughly 0.3 in correlation with the target, in line with the weak single-feature signals of section 8.
* **D11 (Debrecen):** the six microaneurysm counts `ma1` to `ma6` form a block of correlations between about 0.8 and 1.0, and the exudate variables form two further blocks, with `exudate3` and `exudate3.1` almost identical. The correlation with the target does not exceed about 0.3 (`ma1`).
* **D12 (CKD):** `hemo`, `pcv` and `rbcc` are correlated with one another (about 0.7 to 0.85), and several variables correlate strongly with the target (`sg`, `hemo`, `pcv`, `rbcc` negatively, `al` positively), confirming the strong signal of this dataset.
* **Consequence.** Redundant variables do not harm tree-based ensembles in terms of prediction, but they spread the importance measures among correlated features; this must be kept in mind when interpreting feature importances later.

## 11. Summary of data-quality flags

In [ ]:
warn = []
for k in data:
    s = summary.astype(object).loc[k]
    if s.imbalance_ratio >= 3: warn.append((k, f"imbalanced (ratio {s.imbalance_ratio}) -> use F1/AUPRC, stratified splits"))
    if s.missing_pct > 0:      warn.append((k, f"{s.missing_cells} missing cells -> imputation must be inside CV folds"))
    if s.duplicate_rows > 0:   warn.append((k, f"{s.duplicate_rows} duplicate rows ({s.duplicate_pct}%)"))
    if leak.loc[k, "test_rows_with_train_copy_pct"] >= 1: warn.append((k, f"{leak.loc[k,'test_rows_with_train_copy_pct']}% of test rows have a train copy"))
    if s.constant_cols > 0:    warn.append((k, f"{s.constant_cols} constant column(s)"))
    if s.n < 400:              warn.append((k, f"small (n={s.n}) -> high variance, use repeated CV"))
    if auc_tables[k].strength.iloc[0] >= 0.90: warn.append((k, f"feature '{auc_tables[k].feature.iloc[0]}' alone gives strength {auc_tables[k].strength.iloc[0]}"))
    if base.loc[k, "tree_acc"] >= 0.98: warn.append((k, f"a default tree already reaches {base.loc[k,'tree_acc']} accuracy (too easy / leakage?)"))
    if base.loc[k, "forest_acc"] - base.loc[k, "majority_acc"] < 0.05: warn.append((k, "forest barely beats the majority class"))
w = pd.DataFrame(warn, columns=["dataset", "warning"])
w.to_csv(eda.RESULTS / "eda_warnings.csv", index=False)
display(w if len(w) else "No flag raised.")

## 12. Conclusions and consequences for the experimental protocol

| Dataset | Difficulty for the models | Main data issue | Treatment in the experiments |
|---|---|---|---|
| D2 Cleveland | moderate (forest 0.835) | small (n = 303), 6 missing cells | repeated stratified CV, imputation inside folds |
| D5 BUPA | hard (forest 0.722) | label is a convention (`selector`), weak features, n = 345 | repeated CV; results discussed with caution |
| D6 ILPD | hardest (forest ≈ majority rule) | no signal for default models, 2.2 % duplicates | ROC-AUC / AUPRC in addition to F1; tuned models |
| D8 early diabetes | near ceiling | **51.7 % duplicates, 68.9 % test leakage** | compare protocol with and without duplicates |
| D11 Debrecen | hard (forest 0.678) | strongly collinear feature blocks | feature-importance results interpreted with care |
| D12 CKD | easy (forest 0.992) | **10.5 % missing cells in 60 % of patients** | imputation strictly inside folds |
| D16 dermatology | trivial (forest 1.000) | single variable at 0.984, binarised target | kept as a reference; cannot discriminate methods |

Main conclusions:

1. The datasets differ strongly in difficulty. On D16, D12 and D8 near-perfect scores leave no room to distinguish ensemble methods, while D2, D5, D6 and D11 are the datasets on which a comparison is informative.
2. **Duplicate rows are a real source of optimistic bias**, mostly in D8. Experiments are therefore carried out under two protocols (data as downloaded and data without exact duplicates) so that the inflation can be quantified.
3. Imputation and encoding are always fitted on training folds only, which matters most for D12.
4. Because several datasets are small and the fold-to-fold standard deviation reaches 0.04 to 0.07, repeated stratified cross-validation and statistical tests are used to decide whether differences between methods are real.
5. Accuracy is reported together with precision, recall, F1 and the confusion matrix, since the majority-class rule can reach 54 % to 71 % accuracy without detecting a single patient in some datasets.
6. The article's positive class is not always the disease class (D2, D5, D6, D12). Comparisons with the article rely on accuracy and ROC-AUC, which do not depend on that choice; precision, recall and F1 are computed here with the disease as positive class.
7. The experiments must cover all 16 datasets of the article: the remaining binary Kaggle datasets (D1, D3, D4, D7, D9, D10, D13, D14) are added as soon as their files are available, and the multiclass datasets (D15, and D16 in its original six-class form) require macro-averaged metrics.